In [1]:
! pip install pypsa highspy openpyxl "xarray<=2024.9.0"

In [2]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import json
import time
import pypsa
import warnings

ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model")


In [4]:
sys.path.append(PROJECT_DIR)

from modules.getting_input_data import get_input_data, load_model_data

input_file_name = "input_file.xlsx"
input_data = get_input_data(PROJECT_DIR, input_file_name)

File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.


In [5]:
MODEL_DATA_DIR = os.path.join(PROJECT_DIR,str(input_data["data_set"]),
    input_data['model_data_dir'],input_data["project_name"],
    input_data["scenario"], str(input_data["year"]))

RESULTS_DIR = os.path.join(PROJECT_DIR,str(input_data['data_set']),
    input_data['result_dir'],input_data['project_name'],
    input_data['scenario'],str(input_data['year']))

# Importing Model Data

The following cell demonstrates how to initialize the model using only the files saved in `MODEL_DATA_DIR`.

In [6]:
model_data = load_model_data(MODEL_DATA_DIR)

✅ Successfully imported and reconstructed model_data from: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2024/model_data/Europe/GA/2040


# Resampling Data

In [7]:
# Set global resampling frequency
resample_freq = "96h"
resample_freq_hours = float(pd.Timedelta(resample_freq).total_seconds() / 3600)

print(f"Target Resampling: {resample_freq} ({resample_freq_hours} hours)")

Target Resampling: 96h (96.0 hours)


In [8]:

for zone, z_data in model_data.get('Zonal', {}).items():
    for key, val in z_data.items():
        if isinstance(val, pd.Series):
            model_data['Zonal'][zone][key] = val.resample(resample_freq).mean()

# Verification: Check one key from the first zone
first_zone = list(model_data['Zonal'].keys())[0]
new_snapshots = model_data["Zonal"][first_zone]["Rooftop_PV"].index

print(f"New Snapshots Count: {len(new_snapshots)}")

New Snapshots Count: 92


In [9]:
import pandas as pd

# Convert frequency string to float number of hours
resample_freq_hours = float(pd.Timedelta(resample_freq).total_seconds() / 3600)

print(f"Resampling Frequency: {resample_freq}")
print(f"Resampling Hours (float): {resample_freq_hours}")


Resampling Frequency: 96h
Resampling Hours (float): 96.0


# Dual Network Model

In [10]:
warnings.simplefilter(action='ignore', category=FutureWarning)
import json

# 1. INITIALIZE NETWORK
n_dual = pypsa.Network()

# Synchronize snapshots
n_dual.set_snapshots(new_snapshots)
n_dual.snapshot_weightings[:] = resample_freq_hours

# Add Energy Carriers
carriers = ["AC", "H2", "natural_gas", "CO2", "SNG", "e-Kerosene", "e-Diesel",
            "hard_coal", "lignite", "oil", "hydro", "battery", "nuclear", "heat", "solar", "ev"]
for c in carriers:
    n_dual.add("Carrier", c)

prices = model_data["Global"]["prices"]

# Global market for hard coal, lignite, and oil
n_dual.add("Bus", "hard_coal_market", carrier="hard_coal")
n_dual.add("Generator", "Global_Hard_Coal_Import", bus="hard_coal_market", carrier="hard_coal", p_nom_extendable=True, marginal_cost=prices["commodity"]["Hard coal (EUR/t)"])

n_dual.add("Bus", "lignite_market", carrier="lignite")
n_dual.add("Generator", "Global_Lignite_Import", bus="lignite_market", carrier="lignite", p_nom_extendable=True, marginal_cost=prices["commodity"]["Lignite G1"])

n_dual.add("Bus", "oil_market", carrier="oil")
n_dual.add("Generator", "Global_Oil_Import", bus="oil_market", carrier="oil", p_nom_extendable=True, marginal_cost=prices["commodity"].get("Crude oil (EUR/MWh)", 40))

if input_data.get('H2', False) and input_data.get('Synthetic_fuels', False):
    n_dual.add("Bus", "CO2_bus", carrier="CO2")
    n_dual.add("Bus", "SNG_bus", carrier="SNG")
    n_dual.add("Bus", "e-Kerosene_bus", carrier="e-Kerosene")
    n_dual.add("Bus", "e-Diesel_bus", carrier="e-Diesel")

    # Demands (Fixed mapping and removed undefined 'zone' variable)
    eFuel = model_data["Global"]["eFuel"]
    if 'SNG' in eFuel:
        n_dual.add("Load", "Global_SNG_Demand", bus="SNG_bus", p_set=eFuel["SNG"])
    if 'eKerosine' in eFuel:
        n_dual.add("Load", "Global_eKerosene_Demand", bus="e-Kerosene_bus", p_set=eFuel["eKerosine"])
    if 'eDiesel' in eFuel:
        n_dual.add("Load", "Global_eDiesel_Demand", bus="e-Diesel_bus", p_set=eFuel["eDiesel"])

# Regional Buses - Offshore wind farms
if "electricity" in model_data.get("Regional", {}) and "Offshore" in model_data["Regional"]["electricity"]:
    offshore_config = model_data["Regional"]["electricity"]["Offshore"]
    if offshore_config:  # ensure it's not empty
        for offshore_region, params in offshore_config.items():
            bus_elec = f"{offshore_region}_Hub_Elec"
            bus_h2 = f"{offshore_region}_Hub_H2"

            n_dual.add("Bus", bus_elec, carrier="AC")
            n_dual.add("Bus", bus_h2, carrier="H2")


# 2. BUILD FULLY INTEGRATED ZONAL NETWORKS
for zone, z_data in model_data["Zonal"].items():

    # --- BUSES ---
    elec_bus = f"{zone}_electricity_market"
    prosumer_bus = f"{zone}_prosumer_bus"
    gas_bus = f"{zone}_gas_bus"
    CH4_heat_bus = f"{zone}_CH4_heat_bus"
    H2_heat_bus = f"{zone}_H2_heat_bus"

    n_dual.add("Bus", elec_bus, carrier="AC")
    n_dual.add("Bus", prosumer_bus, carrier="AC")
    n_dual.add("Bus", gas_bus, carrier="natural_gas")
    # heats
    n_dual.add("Bus", CH4_heat_bus, carrier="heat")
    n_dual.add("Bus", H2_heat_bus, carrier="heat")

    if input_data.get('H2', False):
        n_dual.add("Bus", f"{zone}_SRES_AC", carrier="AC")
        n_dual.add("Bus", f"{zone}_DRES_AC", carrier="AC")
        n_dual.add("Bus", f"{zone}_h2_zone1", carrier="H2")
        n_dual.add("Bus", f"{zone}_h2_zone2", carrier="H2")

    # --- GAS source ---
    n_dual.add("Generator", f"{zone}_Gas_Source", bus=f"{zone}_gas_bus", carrier="natural_gas", p_nom_extendable=True, marginal_cost=prices["commodity"]['Gas (EUR/MWh)'] if 'Gas (EUR/MWh)' in prices["commodity"] else prices["commodity"].get('gas (EUR/MWh)', 40))

    # --- E L E C T R I C I T Y   M O D E L ---
    # --- Conventional electricity generators ---
    if 'Gas' in z_data:
        gas_val = z_data['Gas']
        if isinstance(gas_val, pd.Series):
            n_dual.add("Link", f"{zone}_Gas_Plant", bus0=f"{zone}_gas_bus", bus1=elec_bus, p_nom=1, p_max_pu=gas_val, efficiency=0.55)
        elif gas_val > 0:
            n_dual.add("Link", f"{zone}_Gas_Plant", bus0=f"{zone}_gas_bus", bus1=elec_bus, p_nom=gas_val, efficiency=0.55)

    if 'Hard coal' in z_data:
        hc_val = z_data['Hard coal']
        if isinstance(hc_val, pd.Series):
            n_dual.add("Link", f"{zone}_Hard_Coal_Plant", bus0="hard_coal_market", bus1=elec_bus, p_nom=1, p_max_pu=hc_val, efficiency=0.40)
        elif hc_val > 0:
            n_dual.add("Link", f"{zone}_Hard_Coal_Plant", bus0="hard_coal_market", bus1=elec_bus, p_nom=hc_val, efficiency=0.40, ramp_limit_up=1/12, ramp_limit_down=1/12)

    if 'Lignite' in z_data:
        lig_val = z_data['Lignite']
        if isinstance(lig_val, pd.Series):
            n_dual.add("Link", f"{zone}_Lignite_Plant", bus0="lignite_market", bus1=elec_bus, p_nom=1, p_max_pu=lig_val, efficiency=0.40)
        elif lig_val > 0:
            n_dual.add("Link", f"{zone}_Lignite_Plant", bus0="lignite_market", bus1=elec_bus, p_nom=lig_val, efficiency=0.40, ramp_limit_up=1/12, ramp_limit_down=1/12)

    if 'Oil' in z_data:
        oil_val = z_data['Oil']
        if isinstance(oil_val, pd.Series):
            n_dual.add("Link", f"{zone}_Oil_Plant", bus0="oil_market", bus1=elec_bus, p_nom=1, p_max_pu=oil_val, efficiency=0.35)
        elif oil_val > 0:
            n_dual.add("Link", f"{zone}_Oil_Plant", bus0="oil_market", bus1=elec_bus, p_nom=oil_val, efficiency=0.35, ramp_limit_up=1/12, ramp_limit_down=1/12)

    if 'Nuclear' in z_data:
        nuc_val = z_data['Nuclear']
        if isinstance(nuc_val, pd.Series):
            n_dual.add("Generator", f"{zone}_Nuclear_Plant", bus=elec_bus, carrier="nuclear", p_nom=1, p_max_pu=nuc_val, marginal_cost=prices["technology"]['nuclear (EUR/MWh)'])
        elif nuc_val > 0:
            n_dual.add("Generator", f"{zone}_Nuclear_Plant", bus=elec_bus, carrier="nuclear", p_nom=nuc_val, marginal_cost=prices["technology"]['nuclear (EUR/MWh)'], ramp_limit_up=1/48, ramp_limit_down=1/48)

    # Variable Renewable Energy Sources
    if 'RoR_MW' in z_data:
        ror_val = z_data['RoR_MW']
        if isinstance(ror_val, pd.Series):
            n_dual.add("Generator", f"{zone}_RoR", bus=elec_bus, carrier="hydro", p_nom=1, p_max_pu=ror_val, marginal_cost=prices['technology'].get('hydro (EUR/MWh)', 0))

    if 'Other_RES' in z_data:
        other_res_val = z_data['Other_RES']
        if isinstance(other_res_val, pd.Series):
            n_dual.add("Generator", f"{zone}_Other_RES", bus=elec_bus, p_nom=1, p_max_pu=other_res_val, marginal_cost=prices['technology']['RES (EUR/MWh)'])

    # Slack generator
    n_dual.add("Generator", f"{zone}_AC_Slack", bus=elec_bus, carrier="AC", p_nom_extendable=True, marginal_cost=1e4)

    if 'Wind_Offshore' in z_data:
        offshore_val = z_data['Wind_Offshore']
        if isinstance(offshore_val, pd.Series):
            raw_regions = input_data.get('regions', {})
            country_map = json.loads(raw_regions) if isinstance(raw_regions, str) else raw_regions

            # Handle country_map as a dictionary or a list
            if isinstance(country_map, dict):
                country_name = next((k for k, v in country_map.items() if zone.startswith(v)), zone[:2])
            else:
                # If it's a list, look for strings that match the start of the zone
                country_name = next((c for c in country_map if zone.startswith(c[:2])), zone[:2])

            matched_regions = [reg for reg, cfg in offshore_config.items() if country_name in cfg.get('onshore_nodes', []) or zone in cfg.get('onshore_nodes', [])]

            if matched_regions:
                split_factor = 1.0 / len(matched_regions)
                for reg in matched_regions:
                    n_dual.add("Generator", f"{zone}_{reg}_Wind_Offshore", bus=f"{reg}_Hub_Elec", p_nom=split_factor, p_max_pu=offshore_val, marginal_cost=prices['technology']['RES (EUR/MWh)'])
                    n_dual.add("Link", f"{zone}_{reg}_Electrolyser_Offshore", bus0=f"{reg}_Hub_Elec", bus1=f"{reg}_Hub_H2",  p_nom_extendable=True, efficiency=z_data.get('elec_efficiency', 0.68))

    # --- Storage Assets ---
    if input_data.get('FLEX', False):
        # 1. Hydro Reservoir
        p_res = z_data.get('Reservoir_P_nom', 0)
        if p_res > 0:
            res_inflow = z_data.get('Res_Inflow_MW', 0)
            n_dual.add("StorageUnit", f"{zone}_HPP_Reservoir", bus=elec_bus, carrier="hydro", p_nom=p_res,
                  max_hours=z_data.get('Reservoir_E_nom', 0) / p_res,
                  cyclic_state_of_charge=True, efficiency_dispatch=0.90,
                  p_max_pu=res_inflow/p_res if isinstance(res_inflow, pd.Series) else 0)

        # 2. Hydro Pondage
        p_pond = z_data.get('Pondage_P_nom', 0)
        if p_pond > 0:
            pond_inflow = z_data.get('Pondage_MW', 0)
            n_dual.add("StorageUnit", f"{zone}_HPP_Pondage", bus=elec_bus, carrier="hydro", p_nom=p_pond,
                  max_hours=z_data.get('Pondage_E_nom', 0) / p_pond,
                  cyclic_state_of_charge=True, efficiency_dispatch=0.90,
                  p_max_pu=pond_inflow/p_pond if isinstance(pond_inflow, pd.Series) else 0)

        # 3. PHS Open
        p_open = z_data.get('PHS_Open_P_nom', 0)
        if p_open > 0:
            p_pump = z_data.get('PHS_Open_P_store', 0)
            n_dual.add("StorageUnit", f"{zone}_PHS_Open", bus=elec_bus, carrier="hydro", p_nom=p_open,
                  p_max_pu_store=p_pump/p_open,
                  max_hours=z_data.get('PHS_Open_E_nom', 0) / p_open,
                  cyclic_state_of_charge=True, efficiency_dispatch=0.90, efficiency_store=0.90)

        # 4. PHS Closed
        p_closed = z_data.get('PHS_Closed_P_nom', 0)
        if p_closed > 0:
            p_pump_c = z_data.get('PHS_Closed_P_store', 0)
            n_dual.add("StorageUnit", f"{zone}_PHS_Closed", bus=elec_bus, carrier="hydro", p_nom=p_closed,
                  p_max_pu_store=p_pump_c/p_closed,
                  max_hours=z_data.get('PHS_Closed_E_nom', 0) / p_closed,
                  cyclic_state_of_charge=True, efficiency_dispatch=0.90, efficiency_store=0.90)

        # 5. Battery
        p_bat = z_data.get('Battery_P_nom', 0)
        if p_bat > 0:
            n_dual.add("StorageUnit", f"{zone}_Battery", bus=elec_bus, carrier="battery", p_nom=p_bat,
                  p_max_pu_store=1.0,
                  max_hours=z_data.get('Battery_E_nom', 0) / p_bat,
                  cyclic_state_of_charge=True, efficiency_dispatch=0.95, efficiency_store=0.95)

    # --- Electricity loads ---
    n_dual.add("Load", f"{zone}_elec_demand", bus=elec_bus, p_set=z_data['El_market'])

    # --- Prosumer Bus ---
      # Links to Main Grid
    n_dual.add("Link", f"{zone}_Grid_Import", bus0=elec_bus, bus1=prosumer_bus, p_nom_extendable=True, efficiency=1.0)
    n_dual.add("Link", f"{zone}_Grid_Export", bus0=prosumer_bus, bus1=elec_bus, p_nom_extendable=True, efficiency=1.0)

      # Demand
    n_dual.add("Load", f"{zone}_Prosumer_Demand", bus=prosumer_bus, p_set=z_data['El_prosumer'])

      # Rooftop PV
    if 'Rooftop_PV' in z_data:
        roof_val = z_data['Rooftop_PV']
        if isinstance(roof_val, pd.Series):
            n_dual.add("Generator", f"{zone}_Rooftop_PV", bus=prosumer_bus, p_nom=1, p_max_pu=roof_val, carrier="solar", marginal_cost=prices['technology']['RES (EUR/MWh)'])

      # Local Battery
    #n_dual.add("StorageUnit", f"{zone}_Prosumer_Battery", bus=prosumer_bus, p_nom_extendable=True, carrier="battery", max_hours=4, efficiency_store=0.92, efficiency_dispatch=0.92)

      # Fixed EV Demand
    n_dual.add("Load", f"{zone}_Fixed_EV_Demand", bus=prosumer_bus, p_set=z_data.get('Fixed_EV_Demand', 0.0))

      # Flexible EV
    #n_dual.add("StorageUnit", f"{zone}_Flexible_EV", bus=prosumer_bus, p_nom_extendable=True, carrier="ev", max_hours=8, efficiency_store=0.95, efficiency_dispatch=0.95)

      # Heat Pump
    n_dual.add("Link", f"{zone}_HP_to_CH4_heat", bus0=prosumer_bus, bus1=CH4_heat_bus, p_nom_extendable=True, efficiency=3.2)
    n_dual.add("Link", f"{zone}_HP_to_H2_heat", bus0=prosumer_bus, bus1=H2_heat_bus, p_nom_extendable=True, efficiency=3.2)

    # --- Gas Heat ---
    if 'CH4_heat' in z_data:
        n_dual.add("Load", f"{zone}_CH4_Heat_Demand", bus=CH4_heat_bus, p_set=z_data['CH4_heat'])
    # Gas boiler
    n_dual.add("Link", f"{zone}_Gas_Boiler", bus0=gas_bus, bus1=CH4_heat_bus, p_nom_extendable=True, efficiency=0.90)

    # --- H Y D R O G E N   M O D E L ---
    if input_data.get('H2', False):
        if z_data.get('SMR_g_zone1', 0) > 0:
            n_dual.add("Link", f"{zone}_SMR_Grey", bus0=f"{zone}_gas_bus", bus1=f"{zone}_h2_zone1", p_nom=z_data['SMR_g_zone1'], efficiency=0.70, marginal_cost=prices["technology"]["SMR (EUR/MWh)"])
        if z_data.get('SMR_b_zone1', 0) > 0:
            n_dual.add("Link", f"{zone}_SMR_Blue", bus0=f"{zone}_gas_bus", bus1=f"{zone}_h2_zone1", p_nom=z_data['SMR_b_zone1'], efficiency=0.65, marginal_cost=prices["technology"]["SMR (EUR/MWh)"])

        if 'SRES' in z_data:
            n_dual.add("Generator", f"{zone}_SRES_plant", bus=f"{zone}_SRES_AC", carrier="AC", p_nom=1, p_max_pu=z_data['SRES'], marginal_cost=prices["technology"]["RES (EUR/MWh)"])
            n_dual.add("Link", f"{zone}_SRES_to_mgrid", bus0=f"{zone}_SRES_AC", bus1=elec_bus, p_nom=1e6, efficiency=1.0)
            n_dual.add("Link", f"{zone}_mgrid_to_SRES", bus0=elec_bus, bus1=f"{zone}_SRES_AC", p_nom=1e6, efficiency=1.0)

        if 'DRES' in z_data:
            n_dual.add("Generator", f"{zone}_DRES_plant", bus=f"{zone}_DRES_AC", carrier="AC", p_nom=1, p_max_pu=z_data['DRES'], marginal_cost=prices["technology"]["RES (EUR/MWh)"])
            n_dual.add("Link", f"{zone}_mgrid_to_DRES", bus0=elec_bus, bus1=f"{zone}_DRES_AC", p_nom=1e6, efficiency=1.0)

        if z_data.get('SRES_electrolyser', 0) > 0:
            n_dual.add("Link", f"{zone}_SRES_Electrolyser", bus0=f"{zone}_SRES_AC", bus1=f"{zone}_h2_zone1",
                       p_nom=z_data['SRES_electrolyser'], efficiency=z_data.get('elec_efficiency', 0.68))
        if z_data.get('DRES_electrolyser', 0) > 0:
            n_dual.add("Link", f"{zone}_DRES_Electrolyser", bus0=f"{zone}_DRES_AC", bus1=f"{zone}_h2_zone2",
                       p_nom=z_data['DRES_electrolyser'], efficiency=z_data.get('elec_efficiency', 0.68))

        if input_data.get('FLEX', True):
            if z_data.get('H2_storage_p_max_z1', 0) > 0:
                n_dual.add("StorageUnit", f"{zone}_H2_Storage_Z1", bus=f"{zone}_h2_zone1", carrier="H2",
                p_nom=z_data['H2_storage_p_max_z1'],
                max_hours=z_data.get('H2_storage_energy_z1', 0)/z_data['H2_storage_p_max_z1'],
                cyclic_state_of_charge=True, efficiency_dispatch=0.97, efficiency_store=0.98)
            if z_data.get('H2_storage_p_max_z2', 0) > 0:
                n_dual.add("StorageUnit", f"{zone}_H2_Storage_Z2", bus=f"{zone}_h2_zone2", carrier="H2",
                p_nom=z_data['H2_storage_p_max_z2'],
                p_max_pu_store=z_data['H2_storage_p_load_z2'] / z_data['H2_storage_p_max_z2'],
                max_hours=z_data['H2_storage_energy_z2'] / z_data['H2_storage_p_max_z2'],
                cyclic_state_of_charge=True, efficiency_dispatch=0.97, efficiency_store=0.98)

        # Hydrogen demand
        if 'H2_zone_1' in z_data:
             n_dual.add("Load", f"{zone}_H2_Demand_Z1", bus=f"{zone}_h2_zone1", p_set=z_data['H2_zone_1'])
             n_dual.add("Generator", f"{zone}_H2_Slack_Z1", bus=f"{zone}_h2_zone1", carrier="H2", p_nom_extendable=True, marginal_cost=1e3)
        if 'H2_zone_2' in z_data:
             n_dual.add("Load", f"{zone}_H2_Demand_Z2", bus=f"{zone}_h2_zone2", p_set=z_data['H2_zone_2'])
             n_dual.add("Generator", f"{zone}_H2_Slack_Z2", bus=f"{zone}_h2_zone2", carrier="H2", p_nom_extendable=True, marginal_cost=1e3)

        # Heat demand
        if 'H2_heat' in z_data:
            n_dual.add("Load", f"{zone}_H2_Heat_Demand", bus=H2_heat_bus, p_set=z_data['H2_heat'])
         # Hydrogen boiler
        n_dual.add("Link", f"{zone}_H2_boiler", bus0=f"{zone}_h2_zone2", bus1=H2_heat_bus, p_nom_extendable=True, efficiency=0.90)


        # H2 --> Electricity
        if 'Hydrogen - CCGT' in z_data:
          H2_CCGT = z_data['Hydrogen - CCGT']
          if isinstance(H2_CCGT, pd.Series):
            n_dual.add("Link", f"{zone}_H2_CCGT", bus0=f"{zone}_h2_zone2", bus1=elec_bus, p_nom=1, p_max_pu=H2_CCGT, efficiency=0.55)
          elif H2_CCGT > 0:
            n_dual.add("Link", f"{zone}_H2_CCGT", bus0=f"{zone}_h2_zone2", bus1=elec_bus, p_nom=H2_CCGT, efficiency=0.55)

        if 'Hydrogen - Fuel Cell' in z_data:
          H2_FC = z_data['Hydrogen - Fuel Cell']
          if isinstance(H2_FC, pd.Series):
            n_dual.add("Link", f"{zone}_H2_FC", bus0=f"{zone}_h2_zone2", bus1=elec_bus, p_nom=1, p_max_pu=H2_FC, efficiency=0.40)
          elif H2_FC > 0:
            n_dual.add("Link", f"{zone}_H2_FC", bus0=f"{zone}_h2_zone2", bus1=elec_bus, p_nom=H2_FC, efficiency=0.40)

        # Synthetic fuels
        if input_data.get('Synthetic_fuels', False):
            n_dual.add("Link", f"{zone}_H2_to_SNG", bus0=f"{zone}_h2_zone2", bus1="SNG_bus", p_nom_extendable=True, efficiency=0.6)
            n_dual.add("Link", f"{zone}_H2_to_eKerosene", bus0=f"{zone}_h2_zone2", bus1="e-Kerosene_bus", p_nom_extendable=True, efficiency=0.6)
            n_dual.add("Link", f"{zone}_H2_to_eDiesel", bus0=f"{zone}_h2_zone2", bus1="e-Diesel_bus", p_nom_extendable=True, efficiency=0.6)

    else:
        if 'Electrolyser_RES_part' in z_data:
            n_dual.add("Generator", f"{zone}_Electrolyser_RES_part", bus=elec_bus, carrier="AC", p_nom=1, p_max_pu=z_data['Electrolyser_RES_part'], marginal_cost=prices["technology"]["RES (EUR/MWh)"])

# --- NTCs LINKS ---
if input_data.get('NTC', True):

  # electricity (internal)
  if "electricity" in model_data.get("Regional", {}) and "NTC" in model_data["Regional"]["electricity"]:
    elec_ntc = model_data["Regional"]["electricity"]["NTC"]
    if isinstance(elec_ntc, dict) and "Internal" in elec_ntc:
      if not isinstance(elec_ntc["Internal"], pd.DataFrame):
        elec_ntc["Internal"] = pd.DataFrame(elec_ntc["Internal"])
      if not elec_ntc["Internal"].empty:
        for index, row in elec_ntc["Internal"].iterrows():
            z1, z2, cap, eff = row['zone 1'], row['zone 2'], row['capacity, MW'], row['losses']
            n_dual.add("Link", f"Elec_Link_{z1}_{z2}", bus0=f"{z1}_electricity_market", bus1=f"{z2}_electricity_market", p_nom=cap, efficiency=eff, bidirectional=False)

  # hydrogen
  if input_data.get('H2', False) and input_data.get('NTC', False):
    if "H2" in model_data.get("Regional", {}) and "NTC" in model_data["Regional"]["H2"]:
      h2_ntc = model_data["Regional"]["H2"]["NTC"]

      # hydrogen (internal)
      if isinstance(h2_ntc, dict) and "Internal" in h2_ntc:
        if not isinstance(h2_ntc["Internal"], pd.DataFrame):
          h2_ntc["Internal"] = pd.DataFrame(h2_ntc["Internal"])
        if not h2_ntc["Internal"].empty:
          for index, row in h2_ntc["Internal"].iterrows():
              z1, z2, cap, eff = row['zone 1'], row['zone 2'], row['capacity, MW'], row['losses']
              n_dual.add("Link", f"H2_Pipe_{z1}_{z2}", bus0=f"{z1}_h2_zone2", bus1=f"{z2}_h2_zone2", p_nom=cap, efficiency=eff, bidirectional=False)

      # hydrogen (import)
      if isinstance(h2_ntc, dict) and "Import" in h2_ntc:
        if not isinstance(h2_ntc["Import"], pd.DataFrame):
          h2_ntc["Import"] = pd.DataFrame(h2_ntc["Import"])
        if not h2_ntc["Import"].empty:
          for index, row in h2_ntc["Import"].iterrows():
            z_from, z_to, cap, price, H2type, fuel = row['zone from'], row['zone to'], row['capacity'], row['price'], row['type'], row['fuel']
            target_bus = f"{z_to}_h2_zone2"
            if fuel == "Pure Hydrogen" :
              n_dual.add("Generator", f"H2_Import_{z_from}_{z_to}", bus=target_bus, carrier="H2", p_nom=cap, marginal_cost=price, efficiency=1.0)
            else :
              n_dual.add("Generator", f"H2_Ammonia_{z_to}", bus=target_bus, carrier="H2", p_nom=cap, marginal_cost=price, efficiency=1.0)

    # offshore wind generatotrs
    for offshore_region, params in offshore_config.items():
        for zone in params.get('onshore_nodes', []):
            # Electricity Link (HVDC)
            n_dual.add("Link", f"HVDC_{offshore_region}_to_{zone}",
                bus0 = f"{offshore_region}_Hub_Elec",
                bus1 = f"{zone}_electricity_market",
                    p_nom_extendable=True)

            # Hydrogen Pipeline Link
            if input_data.get('H2', False):
                n_dual.add("Link", f"H2Pipe_{offshore_region}_to_{zone}",
                    bus0=f"{offshore_region}_Hub_H2",
                    bus1=f"{zone}_h2_zone2",
                    p_nom_extendable=True)
n_dual.sanitize()

In [11]:
import time

# 3. OPTIMIZATION

print(f"Optimizing the {input_data['project_name']} Dual Network (n_dual)... (Full Default Settings)")
start_time = time.time()

# Calling optimize with no solver_options for maximum heuristic efficiency
status = n_dual.optimize(solver_name='highs', store_model=True)

elapsed_time = time.time() - start_time

print(f"\nOptimization Status: {status}")
if status[0] == "ok":
    print(f"Objective Value: {n_dual.objective:.2f} EUR")
    print(f"Solving Time: {elapsed_time:.2f} seconds")

    # Identify where demand is not being met by local supply
    slack_usage = n_dual.generators_t.p.sum()
    deficits = slack_usage[slack_usage.index.str.contains('slack') & (slack_usage > 1e-3)]

    if not deficits.empty:
        print("\n☑☑ Detected Supply Deficits (Slack Usage) in GWh:")
        print(deficits / 1e3)
    else:
        print("\n✅ All demands met without slack generators.")
else:
    print("Model optimization failed.")

Optimizing the Europe Dual Network (n_dual)... (Full Default Settings)


Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 149.97it/s]



Optimization Status: ('ok', 'optimal')
Objective Value: 2461704635295.89 EUR
Solving Time: 23.94 seconds

✅ All demands met without slack generators.


In [12]:
# Count the total number of variables in the optimized model
# In linopy, we iterate through the dataset keys
total_vars = sum(n_dual.model.variables[v].size for v in n_dual.model.variables)
print(f"The total number of variables in the optimized model is: {total_vars:,}")

# Break down variables by type for better detail
print("\nVariable counts by component:")
for name in n_dual.model.variables:
    size = n_dual.model.variables[name].size
    if size > 0:
        print(f"- {name}: {size:,}")

The total number of variables in the optimized model is: 183,313

Variable counts by component:
- Generator-p_nom: 211
- Link-p_nom: 390
- Generator-p: 46,828
- Link-p: 85,928
- StorageUnit-p_dispatch: 16,652
- StorageUnit-p_store: 16,652
- StorageUnit-state_of_charge: 16,652


# Exporting PyPSA Results
There are two main ways to save your results:
1. **Full Network Export**: Saves everything (buses, links, generators, and all time-series results) to a folder.
2. **Selective Export**: Saving specific DataFrames (like prices or power flow) to Excel or CSV.

In [13]:
prefix_results = f"FLEX_{input_data['FLEX']}_NTC_{input_data['NTC']}_H2_{input_data['H2']}_CH4_{input_data['CH4']}_Syn_{input_data['Synthetic_fuels']}"

In [14]:
import os

# Create a results directory
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

print(f"✅ Results directory is ready at {RESULTS_DIR}")
print('Prefix results are:', prefix_results)

✅ Results directory is ready at /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2024/results/Europe/GA/2040
Prefix results are: FLEX_True_NTC_True_H2_True_CH4_False_Syn_False


In [15]:
# Define the specific network export path
network_export_path = os.path.join(RESULTS_DIR, f'{prefix_results}')

# Export the n_II network to the results directory
n_dual.export_to_csv_folder(network_export_path)

print(f"✅ Network successfully saved to: {network_export_path}")

✅ Network successfully saved to: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2024/results/Europe/GA/2040/FLEX_True_NTC_True_H2_True_CH4_False_Syn_False
